**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Foundations of Signal Processing (2)

The sequel [Part 1](./Foundations_of_Signal_Processing_1.ipynb) promised: the z-transform as the discrete world's native language, multirate processing (changing sample rates without lying), the polyphase trick that makes it cheap, and a first meeting with wavelets.

## 1. Pre-requisites

- [Part 1](./Foundations_of_Signal_Processing_1.ipynb) Sessions 3–8.
- [Complex Analysis Lite](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb) — poles, ROC, residues (used throughout Session 1).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The z-Transform & ROC* (~35 min)
**Goal:** master the discrete transform: ROC geometry, stability, and inversion by partial fractions.
**Builds on:** [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S4–S5; [Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb). &nbsp; **Feeds into:** Session 2 (multirate).

---

## 2. The z-Transform

💡 **Intuition.** The z-transform $X(z) = \sum_n x[n] z^{-n}$ is the DTFT with a volume knob: on $z = re^{j\omega}$, it's the DTFT of $x[n] r^{-n}$ — signals too wild for Fourier become tame after exponential damping. The **ROC** records which damping levels work, and it carries real information: the *same* algebraic $X(z)$ with different ROCs describes different signals (causal vs anticausal). Stability = ROC contains the unit circle; causality = ROC extends outward.

**The table you can now derive** (via residues, [Complex Analysis S2](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb)):

| $x[n]$ | $X(z)$ | ROC |
|---|---|---|
| $\delta[n]$ | $1$ | all $z$ |
| $a^n u[n]$ | $\frac{z}{z-a}$ | $|z| > |a|$ |
| $-a^n u[-n-1]$ | $\frac{z}{z-a}$ | $|z| < |a|$ ← same formula, different signal! |
| $r^n \sin(\theta n) u[n]$ | ratio with poles $re^{\pm j\theta}$ | $|z| > r$ |

**Key properties:** delay $x[n-k] \leftrightarrow z^{-k}X(z)$ (why filters are polynomials in $z^{-1}$), convolution ↔ multiplication.

In [2]:
# The ROC is not decoration: one X(z), two signals — only the ROC disambiguates
a = 1.25                                       # pole OUTSIDE the unit circle
n_ax = np.arange(-20, 20)
causal   = np.where(n_ax >= 0, a**np.clip(n_ax,0,None), 0)       # ROC |z|>1.25 → UNSTABLE grows
anticaus = np.where(n_ax < 0, -a**np.clip(n_ax,None,-1), 0)      # ROC |z|<1.25 → stable, anticausal

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].stem(n_ax, causal); axes[0].set_title("ROC |z|>|a|: causal, blows up")
axes[1].stem(n_ax, anticaus); axes[1].set_title("ROC |z|<|a|: stable, but anticausal")
plt.tight_layout(); plt.show()
print("same X(z) = z/(z−1.25). Stability and causality are a PAIR you choose between when a pole is outside the circle.")

same X(z) = z/(z−1.25). Stability and causality are a PAIR you choose between when a pole is outside the circle.


/tmp/ipykernel_2024933/757155337.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *Multirate: Decimation & Interpolation* (~40 min)
**Goal:** change sample rates honestly: anti-alias before dropping, filter after stuffing.
**Builds on:** Session 1; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (sampling). &nbsp; **Feeds into:** Session 3 (polyphase).

---

## 3. Changing the Sample Rate

💡 **Intuition.** **Downsampling** by $M$ (keep every $M$-th sample) stretches the spectrum by $M$ — anything beyond the new Nyquist folds back as aliasing, so you must low-pass *first* (decimation = filter + downsample). **Upsampling** by $L$ (insert $L-1$ zeros) compresses the spectrum and reveals $L-1$ spectral *images* — ghosts of the original — which the interpolation filter must erase. Every resampler, DAC, and neural 'stride/transposed conv' is these two moves.

In [3]:
fs = 1000
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*40*t) + 0.6*np.sin(2*np.pi*380*t)     # 40 Hz wanted + 380 Hz intruder
M = 4                                                     # target fs = 250 → new Nyquist 125 Hz

naive = x[::M]                                            # just drop samples
proper = sig.decimate(x, M, ftype="fir")                  # anti-alias filter, THEN drop

def spec(y, fs_y):
    f = np.fft.rfftfreq(len(y), 1/fs_y)
    return f, np.abs(np.fft.rfft(y)) / len(y)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8), sharey=True)
for ax, (y, title) in zip(axes, [(naive, "naive [::4] — 380 Hz aliases to 130 Hz!"),
                                  (proper, "decimate() — intruder removed first")]):
    f, S = spec(y, fs/M)
    ax.plot(f, S); ax.set_title(title); ax.set_xlabel("Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2024933/733753377.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# Upsampling: zero-stuffing creates images; the interpolation filter erases them
L = 4
x40 = np.sin(2*np.pi*40*np.arange(0, 1, 1/250))          # a 250 Hz-rate signal
stuffed = np.zeros(len(x40)*L); stuffed[::L] = x40
interp = sig.resample_poly(x40, L, 1)                     # proper polyphase interpolation

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8), sharey=True)
for ax, (y, title) in zip(axes, [(stuffed, "zero-stuffed: 3 spectral images"),
                                  (interp, "after interpolation filter")]):
    f, S = spec(y, 1000)
    ax.plot(f, S); ax.set_title(title); ax.set_xlabel("Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2024933/4267586221.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *Polyphase Structures* (~35 min)
**Goal:** never compute what you'll throw away: the decomposition behind every efficient resampler.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (wavelets).

---

## 4. The Polyphase Trick

💡 **Intuition.** Filter-then-downsample wastes $\frac{M-1}{M}$ of its work computing outputs that get discarded. The polyphase fix: split the filter into $M$ interleaved sub-filters (its *phases*), run each at the **low** rate on the input's interleaved streams, and sum. Identical output, $M\times$ cheaper — pure bookkeeping, no approximation. This structure is why sample-rate conversion is cheap enough to be everywhere, and it's the skeleton of the filter banks in Session 4.

In [5]:
# Polyphase decimation by hand — verify exact equivalence, then time it
h = sig.firwin(64, 1/M)                     # anti-alias filter for M=4
x_long = rng.standard_normal(2**18)

# reference: full-rate filter, then discard 3 of every 4 outputs
ref = np.convolve(x_long, h, "full")[::M]

# scipy's polyphase engine computes ONLY the surviving outputs
pp = sig.upfirdn(h, x_long, up=1, down=M)
print("max |polyphase − reference| =", np.abs(pp - ref[:len(pp)]).max())

import time
tic = time.perf_counter(); _ = np.convolve(x_long, h, "full")[::M]; t_ref = time.perf_counter() - tic
tic = time.perf_counter(); _ = sig.upfirdn(h, x_long, up=1, down=M); t_pp = time.perf_counter() - tic
print(f"full-rate then discard: {t_ref*1e3:.1f} ms   polyphase upfirdn: {t_pp*1e3:.1f} ms   ({t_ref/t_pp:.1f}x)")

max |polyphase − reference| = 1.5543122344752192e-15
full-rate then discard: 2.0 ms   polyphase upfirdn: 1.2 ms   (1.7x)


---
### 🕐 Session 4 of 4 — *Wavelets, a First Meeting* (~40 min)
**Goal:** trade the STFT's fixed window for scale: the Haar transform, coded from scratch.
**Builds on:** Session 3; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (uncertainty).

---

## 5. Beyond Fixed Windows

💡 **Intuition.** The STFT slices time with ONE window length — so it resolves either the click or the pitch well, never both ([Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb)'s uncertainty trade, frozen in). Wavelets spend the uncertainty budget *adaptively*: short windows for high frequencies, long for low — constant-Q tiling. Implementation-wise a wavelet transform is just a two-channel filter bank (low-pass + high-pass, downsample by 2) applied **recursively to the low-pass branch** — Session 3's machinery, iterated.

In [6]:
# The Haar wavelet transform from scratch: averages & differences, recursively
def haar_forward(x):
    out, approx = [], x.astype(float)
    while len(approx) > 1:
        a = (approx[0::2] + approx[1::2]) / np.sqrt(2)      # low-pass  + ↓2
        d = (approx[0::2] - approx[1::2]) / np.sqrt(2)      # high-pass + ↓2
        out.append(d); approx = a
    out.append(approx)
    return out[::-1]                                         # coarsest first

def haar_inverse(coeffs):
    approx = coeffs[0]
    for d in coeffs[1:]:
        up = np.empty(2*len(d))
        up[0::2] = (approx + d) / np.sqrt(2)
        up[1::2] = (approx - d) / np.sqrt(2)
        approx = up
    return approx

# a piecewise-constant "blocks" signal — Fourier's nightmare, wavelets' lunch
t = np.linspace(0, 1, 512)
x = np.select([t < 0.2, t < 0.45, t < 0.6, t < 0.8], [0.0, 1.6, 0.4, 2.0], default=0.9)
x = x + 0.02 * rng.standard_normal(512)
coeffs = haar_forward(x)
print("perfect reconstruction:", np.allclose(haar_inverse(coeffs), x))

perfect reconstruction: True


In [7]:
# Compression bake-off at equal budget: keep the 40 largest coefficients
def keep_top(vals, k):
    flat = np.concatenate(vals) if isinstance(vals, list) else vals
    thresh = np.sort(np.abs(flat))[-k]
    return thresh

k = 40
# wavelet: threshold across all detail levels
flatc = np.concatenate(coeffs)
th_w = np.sort(np.abs(flatc))[-k]
coeffs_c = [np.where(np.abs(c) >= th_w, c, 0) for c in coeffs]
x_wav = haar_inverse(coeffs_c)
# Fourier: keep top-k magnitude bins (hermitian pairs counted once)
X = np.fft.rfft(x)
th_f = np.sort(np.abs(X))[-k//2]
x_fft = np.fft.irfft(np.where(np.abs(X) >= th_f, X, 0), n=len(x))

plt.figure(figsize=(9, 3))
plt.plot(t, x, "k", alpha=0.35, label="signal")
plt.plot(t, x_fft, label=f"Fourier, {k} coeffs (ringing at the step)")
plt.plot(t, x_wav, label=f"Haar, {k} coeffs (step preserved)")
plt.legend(); plt.title("Same budget, different bases — transients favor wavelets")
plt.tight_layout(); plt.show()
print(f"RMSE  Fourier {np.std(x_fft - x):.4f}   Haar {np.std(x_wav - x):.4f}")

RMSE  Fourier 0.1387   Haar 0.0178


/tmp/ipykernel_2024933/487303954.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

The z-transform's ROC settles stability vs causality; decimation and interpolation change rates honestly; polyphase makes it all nearly free; and wavelets re-spend the uncertainty budget where the signal needs it. This is the toolkit of every modern codec and SDR front-end.

---
## Where next

- [Software-Defined Radio](./Software_Defined_Radio.ipynb) — multirate chains in the wild.
- [Compressed Sensing](./Compressed_Sensing.ipynb) — sparsity in a basis, weaponized.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — the STFT/wavelet trade on real sound.